In [2]:
import sys
sys.path.append("..")

In [3]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [4]:
def get_model_adv_pga(X_0, X_r, cfr, alpha, lamb, pga_max_iter: int = 100):
    X_0 = torch.tensor(np.stack(X_0)).float()
    X_r = torch.tensor(np.stack(X_r)).float()
    
    loss_fn = torch.nn.BCELoss(reduction='mean')
    cfr_adv = deepcopy(cfr)
    optimizer = optim.Adam(cfr_adv.parameters(), maximize=True)
    weights_min = [cfr.fc1.weight.data-alpha, cfr.fc2.weight.data-alpha, cfr.fc3.weight.data-alpha, cfr.out.weight.data-alpha]
    weights_max = [cfr.fc1.weight.data+alpha, cfr.fc2.weight.data+alpha, cfr.fc3.weight.data+alpha, cfr.out.weight.data+alpha]
    bias_min = [cfr.fc1.bias.data-alpha, cfr.fc2.bias.data-alpha, cfr.fc3.bias.data-alpha, cfr.out.bias.data-alpha]
    bias_max = [cfr.fc1.bias.data+alpha, cfr.fc2.bias.data+alpha, cfr.fc3.bias.data+alpha, cfr.out.bias.data+alpha]
        
    loss = torch.tensor(1.)
    loss_diff = 1
    i = 0
    # while loss_diff > 1e-4:
    for epoch in range(pga_max_iter):
        prev_loss = loss.clone().detach()
        optimizer.zero_grad()
        
        f_x = cfr_adv(X_r)
        y_target = torch.ones(f_x.shape).float()
        bce_loss = loss_fn(f_x, y_target)
        cost = torch.dist(X_r, X_0, 1)
        loss = bce_loss + lamb*cost
        
        loss.backward()
        optimizer.step()
        
        loss_diff = torch.dist(prev_loss, loss, 1)
        i += 1
        
        # clamp model parameters to -alpha, alpha range
        cfr_adv.fc1.weight.data = cfr_adv.fc1.weight.data.clamp(weights_min[0], weights_max[0])
        cfr_adv.fc2.weight.data = cfr_adv.fc2.weight.data.clamp(weights_min[1], weights_max[1])
        cfr_adv.fc3.weight.data = cfr_adv.fc3.weight.data.clamp(weights_min[2], weights_max[2])
        cfr_adv.out.weight.data = cfr_adv.out.weight.data.clamp(weights_min[3], weights_max[3])
        
        cfr_adv.fc1.bias.data = cfr_adv.fc1.bias.data.clamp(bias_min[0], bias_max[0])
        cfr_adv.fc2.bias.data = cfr_adv.fc2.bias.data.clamp(bias_min[1], bias_max[1])
        cfr_adv.fc3.bias.data = cfr_adv.fc3.bias.data.clamp(bias_min[2], bias_max[2])
        cfr_adv.out.bias.data = cfr_adv.out.bias.data.clamp(bias_min[3], bias_max[3])
    
    wnorms = [
        torch.dist(cfr.fc1.weight.data, cfr_adv.fc1.weight.data, torch.inf),
        torch.dist(cfr.fc2.weight.data, cfr_adv.fc2.weight.data, torch.inf),
        torch.dist(cfr.fc3.weight.data, cfr_adv.fc3.weight.data, torch.inf),
        torch.dist(cfr.out.weight.data, cfr_adv.out.weight.data, torch.inf),
    ]
    
    bnorms = [
        torch.dist(cfr.fc1.bias.data, cfr_adv.fc1.bias.data, torch.inf),
        torch.dist(cfr.fc2.bias.data, cfr_adv.fc2.bias.data, torch.inf),
        torch.dist(cfr.fc3.bias.data, cfr_adv.fc3.bias.data, torch.inf),
        torch.dist(cfr.out.bias.data, cfr_adv.out.bias.data, torch.inf),
    ]
    
    # print(f'Final Loss: {loss}')
    # print(f'Num Iterations: {i}')
    # print(f'weights_alpha, bias_alpha: {max(wnorms), max(bnorms)}')
            
    return cfr_adv

In [24]:
data = pd.read_pickle("../results/recourse/lr_german_alg1_0.001_0.5_4.pkl")
X_0 = np.stack(data['x_0'])
X_r = np.stack(data['x_r'])
theta_0 = data['theta_0']

In [25]:
cfr = Net0(X_0.shape[1])

In [26]:
cfr_new = NN(X_0.shape[1])

In [ ]:
cfr_new.model.load_state_dict()

Sequential(
  (0): Linear(in_features=7, out_features=50, bias=True)
  (1): ReLU()
  (2): Linear(in_features=50, out_features=100, bias=True)
  (3): ReLU()
  (4): Linear(in_features=100, out_features=200, bias=True)
  (5): ReLU()
  (6): Linear(in_features=200, out_features=1, bias=True)
  (7): Sigmoid()
)